# Few-Shot (Multi-Shot) Prompting

few-shot prompting is passing mock fixtures, unit test cases, or input-output examples directly into your function call to prove expected behavior.

When zero-shot prompts fail because the model misunderstands your desired formatting, style, or business logic, few-shot prompting is your primary engineering tool to fix it.

## Phase 1: The Core Mechanics of Few-Shot Prompting
**What is it?**
Few-shot prompting means providing a small set of explicit Input $\rightarrow$ Output examples inside your prompt before passing the actual real-world query.

**Why it Works (The In-Context Learning Phenomenon)**
Remember how LLMs work: they predict the next most likely token based on patterns in the text.
When you provide examples, you are actively writing a pattern into the context window.
The model's attention mechanism recognizes the pattern (e.g., Input format matches Output format) and naturally continues that exact pattern for the final test case.

## Phase 2: Anatomy of a Few-Shot Prompt
A well-structured few-shot prompt uses clear delimiters and labels so the model doesn't confuse your training examples with the actual user request.

[System Instruction]
Extract key entity tags from customer support tickets. Follow the exact formatting style shown in the examples.

[Example 1]
Input: "My billing receipt #4029 was charged twice for Pro subscription."
Output: {"category": "Billing", "priority": "High", "account_type": "Pro"}

[Example 2]
Input: "How do I reset my password? The link expired."
Output: {"category": "Auth", "priority": "Medium", "account_type": "Unknown"}

[Actual User Query]
Input: "The mobile app crashes every time I upload a PNG larger than 5MB."
Output:

### Key Structural Elements:
**Clear Boundaries:** Using labels like Input: and Output: or XML tags (<example>, </example>) ensures the model cleanly separates training data from execution data.

**Consistency:** Every example should follow the exact same syntax (e.g., if Example 1 uses a JSON object, all examples must use JSON).

**Edge Case Inclusion:** If your task has tricky edge cases, dedicate one of your few-shot examples to demonstrating how to handle that specific edge case.

## Phase 3: The Engineering Trade-offs (When to Use Few-Shot)
While few-shot prompting drastically improves accuracy, it comes with architectural trade-offs that software engineers must manage:

**Token Inflation & Cost:** Every example you add consumes context window tokens. If you pass 10 large examples with every API request, your input token cost scales linearly.

**The Sweet Spot:** Research and industry standards show that 3 to 5 examples is usually the optimal balance. Anything past 5 yields diminishing returns.

**Context Window Pollution:** If your examples are outdated or poorly formatted, they can actively mislead the model.

## Phase 4: Advanced Pattern — Dynamic Few-Shot Retrieval
In production applications, hardcoding the exact same 3 examples into your prompt code is often inefficient. If your app handles 50 different categories of user queries, sending billing examples for a technical bug report wastes tokens.

**The Solution:**

Store a database of 50 to 100 high-quality, diverse example pairs.

When a user submits a query, use a Vector Database and embeddings to find the 3 examples in your database that are semantically most similar to the user's query.

Dynamically inject only those 3 relevant examples into the prompt template at runtime.

## Phase 5: Python Code Implementation
Here is how you implement a clean few-shot prompt structure in a production Python function:

In [ ]:
from openai import OpenAI

client = OpenAI()

def classify_support_ticket(ticket_text: str) -> str:
    # Few-shot examples structured cleanly
    few_shot_prompt = """Classify the following customer support ticket into a JSON object with category and priority.

Example 1:
Input: "I was charged twice for my monthly plan!"
Output: {"category": "Billing", "priority": "High"}

Example 2:
Input: "Where can I find the dark mode toggle?"
Output: {"category": "UI/UX", "priority": "Low"}

Example 3:
Input: "The database connection drops under heavy load."
Output: {"category": "Backend", "priority": "Urgent"}

---
Now classify this ticket:
Input: "{ticket_text}"
Output:"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are a precise data classification microservice."},
            {"role": "user", "content": few_shot_prompt.format(ticket_text=ticket_text)}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# Test the function
print(classify_support_ticket("API keys are returning 401 unauthorized errors across all services."))